Подготовка окружения

In [ ]:
%pip install ultralytics pyyaml

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Разбиение dataset'a

In [ ]:
from pathlib import Path
import random
import shutil
import yaml


root = Path("/content/drive/MyDrive/Tabmed/ML/datasets/dataset1")
out = Path("/content/drive/MyDrive/Tabmed/ML/datasets/dataset1_split")

random.seed(42)

images = sorted(
    p for p in (root / "images").iterdir()
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

random.shuffle(images)
split = int(len(images) * 0.8)

train_images = images[:split]
val_images = images[split:]

for part, files in [("train", train_images), ("val", val_images)]:
    img_dir = out / "images" / part
    lbl_dir = out / "labels" / part
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for img in files:
        shutil.copy2(img, img_dir / img.name)
        label = root / "labels" / f"{img.stem}.txt"
        shutil.copy2(label, lbl_dir / label.name)

data = {
    "path": str(out),
    "train": "images/train",
    "val": "images/val",
    "names": {0: "tablet", 1: "capsule"}
}

with open(out / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)

print("Train:", len(train_images), "изображений")
print("Val:", len(val_images), "изображений")
print("YAML:")
print((out / "data.yaml").read_text())

Обучение yolo26n

In [ ]:
from ultralytics import YOLO
from pathlib import Path

data_yaml = (
    "/content/drive/MyDrive/Tabmed/ML/"
    "datasets/dataset1_split/data.yaml"
)

model = YOLO("yolo26n.pt")

results = model.train(
    data=data_yaml,
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,
    project="/content/drive/MyDrive/Tabmed/ML/experiments",
    name="tablet_detection_v1",
    exist_ok=True,
    patience=15,
    seed=42
)

print("Обучение завершено")
print("Папка результатов:")
print(results.save_dir)

Проверка

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from PIL import Image
from IPython.display import display

model_path = (
    "/content/drive/MyDrive/Tabmed/ML/"
    "experiments/tablet_detection_v1/weights/best.pt"
)

images_dir = Path(
    "/content/drive/MyDrive/Tabmed/ML/"
    "datasets/dataset1/images"
)

model = YOLO(model_path)

results = model.predict(
    source=str(images_dir),
    imgsz=640,
    conf=0.25,
    device=0,
    verbose=False
)

print(f"Всего обработано фотографий: {len(results)}")

for result in results[:12]: # можно увеличть количество показываемых файлов
    print(Path(result.path).name)
    display(Image.fromarray(result.plot()[..., ::-1]))